# **House Price Prediction using Machine Learning**

### **Import Important Libraries**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score
from sklearn.tree import plot_tree

### **Read Files**

In [3]:
df = pd.read_csv("train.csv")

In [4]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [5]:
df.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [6]:
# Shape of the dataset
df.shape

(1460, 81)

In [7]:
# Extracted Input And Output
x = df.drop('SalePrice', axis = 1)
y = df['SalePrice']

In [8]:
x.head(2)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal


In [9]:
y.head(2)

0    208500
1    181500
Name: SalePrice, dtype: int64

In [10]:
# Sepearte Numerica And Categorical Columns in Input
num_col = x.select_dtypes(include = ['int64', 'float64']).columns
cat_col = x.select_dtypes(include = ['object']).columns

C:\Users\Sushil Kumar\AppData\Local\Temp\ipykernel_12636\1075969624.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_col = x.select_dtypes(include = ['object']).columns


In [11]:
num_col

Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
       'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF',
       'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea',
       'MiscVal', 'MoSold', 'YrSold'],
      dtype='str')

In [12]:
cat_col

Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='str')

# Preprocessing 

In [13]:
# Apply SimpleImputer and OneHotEncoding using Pipeline
num_trf = Pipeline([('imputer', SimpleImputer(strategy = 'median'))])
cat_trf = Pipeline([('imputer', SimpleImputer(strategy = 'most_frequent')),
                   ('encoder', OneHotEncoder(handle_unknown = 'ignore'))])

# Used Column Transformer 
preprocess = ColumnTransformer([
    ('num', num_trf, num_col),
    ('cat', cat_trf, cat_col)
])

# Model Train

## **RandomForestRegressor**

In [14]:
# Apply the Transformer and Model using Pipeline
model1 = Pipeline([
    ('preprocessor', preprocess),
    ('regressor', RandomForestRegressor(n_estimators = 100))
])

# model1.fit(x,y)

# Evaluation

In [15]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [16]:
x_train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
254,255,20,RL,70.0,8400,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
1066,1067,60,RL,59.0,7837,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2009,WD,Normal
638,639,30,RL,67.0,8777,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,5,2008,WD,Normal
799,800,50,RL,60.0,7200,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,6,2007,WD,Normal
380,381,50,RL,50.0,5000,Pave,Pave,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal


In [17]:
y_train.head()

254     145000
1066    178000
638      85000
799     175000
380     127000
Name: SalePrice, dtype: int64

In [18]:
# Model Trained
model1.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

In [19]:
# Predict Model
y_pred = model1.predict(x_test)

In [20]:
# R2 Score
r2_score(y_test, y_pred)

0.8941585743362106

In [21]:
# Cross Validation using r2
score = cross_val_score(model1, x, y, cv = 10, scoring = 'r2')
print('r2 score:', score.mean())

r2 score: 0.8605063119043084


In [22]:
# cross Validation using Root Mean Square error
score = cross_val_score(model1, x, y, cv = 10, scoring = 'neg_root_mean_squared_error')
print('root_mean_square_error:', -score.mean())

root_mean_square_error: 28791.519123504822


### Results:
Test R² Score = 0.8861
Cross Validation R² Score = 0.8603
RMSE = 28,816

### Conclusion:
The Random Forest Regressor performed well and explained approximately 88.6% of the variation in house prices on the test data. The cross-validation score is also close to the test score, indicating good generalization and low overfitting. However, the prediction error (RMSE) is higher than the best-performing model in this project. Overall, Random Forest provides a strong and reliable baseline for house price prediction.

## **XGBRegressor(Extream Gradient Boosting Regressor)**

In [23]:
from xgboost import XGBRegressor

In [24]:
# Apply the Transformer and Model using Pipeline
model2 = Pipeline([
    ('preprocessor', preprocess),
    ('regressor', XGBRegressor(n_estimators=500,learning_rate=0.5,max_depth=4,random_state=42))])

In [25]:
# Train Model
model2.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

In [26]:
# Predict Model
y_pred2 = model2.predict(x_test)

In [27]:
# Calculate R2 Score
print('r2 score:', r2_score(y_test, y_pred2))

r2 score: 0.905053973197937


In [28]:
# Cross Validation using R2
score = cross_val_score(model2, x, y, cv = 10, scoring = 'r2')
print('Cross validation of r2 score:', score.mean())

Cross validation of r2 score: 0.8510893106460571


In [29]:
# Cross Validation using Root Mean Square Error
score = cross_val_score(model2, x, y, cv = 10, scoring = 'neg_root_mean_squared_error')
print('root_mean_square_error:', -score.mean())

root_mean_square_error: 30175.69609375


### Results:
Test R² Score = 0.9051
Cross Validation R² Score = 0.8511
RMSE = 30,175
### Conclusion:

The XGBoost model achieved a high test R² score of 90.5%, showing excellent predictive capability on the test set. However, its cross-validation score is noticeably lower than the test score, suggesting some degree of overfitting. Additionally, the RMSE is higher than both Random Forest and Gradient Boosting. While XGBoost captures complex relationships effectively, the current hyperparameter settings may require further tuning for better generalization.

## **LinearRegression**

In [30]:
from sklearn.linear_model import LinearRegression

In [31]:
# Apply the Transformer and Model using Pipeline
model3 = Pipeline([
    ('preprocessor', preprocess),
    ('regressor', LinearRegression())])

In [32]:
# Train Model
model3.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

In [33]:
# Predict Model
y_pred3 = model3.predict(x_test)

In [34]:
# Calculate R2 Score
print('r2 score',r2_score(y_test, y_pred3))

r2 score 0.8723736855373118


In [35]:
# Cross Validation using R2
score = cross_val_score(model3, x, y, cv = 10, scoring = 'r2')
print('Cross validation of r2 score:', score.mean())

Cross validation of r2 score: 0.8355781409713791


In [36]:
# Cross Validation using Root Mean Square Error
score = cross_val_score(model3, x, y, cv = 10, scoring = 'neg_root_mean_squared_error')
print('root_mean_square_error:', -score.mean())

root_mean_square_error: 31344.34114288269


### Results:
Test R² Score = 0.8724
Cross Validation R² Score = 0.8356
RMSE = 31,344
### Conclusion:

Linear Regression achieved the lowest performance among all tested models. It explained approximately 87.2% of the variance in house prices but produced the highest RMSE value. Since house price data contains complex non-linear relationships, the simple linear assumptions of this model limit its predictive accuracy. Nevertheless, it serves as a useful benchmark and offers excellent interpretability.

## **GradientBoostingRegressor**

In [37]:
from sklearn.ensemble import GradientBoostingRegressor

In [38]:
# Apply the Transformer and Model using Pipeline
model4 = Pipeline([
    ('preprocessor', preprocess),
    ('regressor', GradientBoostingRegressor(n_estimators=200,learning_rate=0.1,max_depth=3,random_state=42))
])

In [39]:
# Train Model
model4.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

In [40]:
# Predict Model
y_pred4 = model4.predict(x_test)

In [41]:
# Calculate R2 Score
print('r2 score',r2_score(y_test, y_pred4))

r2 score 0.9083304744363973


In [42]:
# Cross validation using R2 Score
score = cross_val_score(model4, x, y, cv = 10, scoring = 'r2')
print('Cross validation of r2 score:', score.mean())

Cross validation of r2 score: 0.8938978213184106


In [43]:
# Cross Validation Using Root Mean Square Error
score = cross_val_score(model4, x, y, cv = 10, scoring = 'neg_root_mean_squared_error')
print('root_mean_square_error:', -score.mean())

root_mean_square_error: 25244.03997899418


### Results:
Test R² Score = 0.9083
Cross Validation R² Score = 0.8939
RMSE = 25,244
### Conclusion:

Gradient Boosting Regressor delivered the best overall performance in this project. It achieved the highest test R² score (90.83%) and the highest cross-validation R² score (89.39%). It also produced the lowest RMSE (25,244), indicating the smallest prediction error among all models. The close alignment between test and cross-validation scores demonstrates strong generalization and robustness.

### Final Conclusion:

This project successfully developed a house price prediction model using machine learning techniques. Among all the models tested, Gradient Boosting Regressor achieved the best performance with the highest R² score (90.83%) and the lowest RMSE (25,244). Therefore, it was selected as the final model for predicting house prices accurately and reliably. The results demonstrate that ensemble learning methods are highly effective for house price prediction tasks.

In [44]:
import joblib

joblib.dump(model4, "model.pkl")

['model.pkl']

In [45]:
import sklearn
print(sklearn.__version__)

1.8.0


In [46]:
# Check Model name
print(model4.named_steps.keys())

dict_keys(['preprocessor', 'regressor'])


In [47]:
print(model4.named_steps)

{'preprocessor': ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath',...
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='str'))]), 'regressor': GradientBoostingRegr

In [48]:
# Find out Important Features to predict saleprize
gb_model = model4.named_steps['regressor'] 
importances = gb_model.feature_importances_

feature_names = x.columns

for feature, importance in sorted(
    zip(feature_names, importances),
    key=lambda x: x[1],
    reverse=True
):
    print(feature, importance)

LotArea 0.5067051613358886
HouseStyle 0.14109759981632883
MasVnrArea 0.0435570349647816
Neighborhood 0.04060104885347649
Utilities 0.03311546918206506
Condition2 0.028904925207671215
Condition1 0.028582529229475177
LotFrontage 0.018332767795049634
Alley 0.014807756407245775
Exterior2nd 0.010730772239820371
LotShape 0.008270784032985876
YearBuilt 0.006940277376059966
Street 0.00693777386420355
MasVnrType 0.00536361764938826
ExterQual 0.004814668673164735
PoolArea 0.004066081017538091
Foundation 0.0038587306320328227
BsmtExposure 0.003000834712077558
CentralAir 0.001772774282181323
ScreenPorch 0.0015017514518909816
MSZoning 0.0013146571060542482
LotConfig 0.0012518156073592361
BsmtFinType2 0.0012075722824753594
LandSlope 0.001163355812947457
BsmtFinType1 0.0011561045551786546
YearRemodAdd 0.00108765165167745
Id 0.0009517993636571491
OverallQual 0.0009437296541313127
ExterCond 0.0008973863434541903
FireplaceQu 0.0007028328222953371
Exterior1st 0.0006316618125716444
RoofStyle 0.00061986854

# these are the top 15 Features which is most important for predict salesprice 
# 1-LotArea 0.5067051613358886
# 2-HouseStyle 0.14109759981632883
# 3-MasVnrArea 0.0435570349647816
# 4-Neighborhood 0.04060104885347649
# 5-Utilities 0.03311546918206506
# 6-Condition2 0.028904925207671215
# 7-Condition1 0.028582529229475177
# 8-LotFrontage 0.018332767795049634
# 9-Alley 0.014807756407245775
# 10-Exterior2nd 0.010730772239820371
# 11-LotShape 0.008270784032985876
# 12-YearBuilt 0.006940277376059966
# 13-Street 0.00693777386420355
# 14-MasVnrType 0.00536361764938826
# 15-ExterQual 0.004814668673164735

In [49]:
df.head(2)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500


In [50]:
# I will take top 15 feature to predict SalePrice
new_x = df[['LotArea','HouseStyle','MasVnrArea','Neighborhood','Utilities','Condition2','Condition1','LotFrontage','Alley','Exterior2nd',
           'LotShape','YearBuilt','Street','MasVnrType','ExterQual']]
new_y = df['SalePrice']

In [51]:
new_x.head()

,LotArea,HouseStyle,MasVnrArea,Neighborhood,Utilities,Condition2,Condition1,LotFrontage,Alley,Exterior2nd,LotShape,YearBuilt,Street,MasVnrType,ExterQual
0,8450,2Story,196.0,CollgCr,AllPub,Norm,Norm,65.0,NaN,VinylSd,Reg,2003,Pave,BrkFace,Gd
1,9600,1Story,0.0,Veenker,AllPub,Norm,Feedr,80.0,NaN,MetalSd,Reg,1976,Pave,NaN,TA
2,11250,2Story,162.0,CollgCr,AllPub,Norm,Norm,68.0,NaN,VinylSd,IR1,2001,Pave,BrkFace,Gd
3,9550,2Story,0.0,Crawfor,AllPub,Norm,Norm,60.0,NaN,Wd Shng,IR1,1915,Pave,NaN,TA
4,14260,2Story,350.0,NoRidge,AllPub,Norm,Norm,84.0,NaN,VinylSd,IR1,2000,Pave,BrkFace,Gd


In [52]:
new_y.head()

0    208500
1    181500
2    223500
3    140000
4    250000
Name: SalePrice, dtype: int64

In [53]:
# Check The null Value
new_x.isnull().sum()

LotArea            0
HouseStyle         0
MasVnrArea         8
Neighborhood       0
Utilities          0
Condition2         0
Condition1         0
LotFrontage      259
Alley           1369
Exterior2nd        0
LotShape           0
YearBuilt          0
Street             0
MasVnrType       872
ExterQual          0
dtype: int64

In [54]:
new_y.isnull().sum()

np.int64(0)

In [55]:
# Categories numerical and categorical columns 
new_num_col = new_x.select_dtypes(include = ['int64', 'float64']).columns
new_cat_col = new_x.select_dtypes(include = ['object']).columns

C:\Users\Sushil Kumar\AppData\Local\Temp\ipykernel_12636\4226344384.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  new_cat_col = new_x.select_dtypes(include = ['object']).columns


In [56]:
new_num_col

Index(['LotArea', 'MasVnrArea', 'LotFrontage', 'YearBuilt'], dtype='str')

In [57]:
new_cat_col

Index(['HouseStyle', 'Neighborhood', 'Utilities', 'Condition2', 'Condition1',
       'Alley', 'Exterior2nd', 'LotShape', 'Street', 'MasVnrType',
       'ExterQual'],
      dtype='str')

In [58]:
# Train test split
new_x_train, new_x_test, new_y_train, new_y_test = train_test_split(new_x, new_y, test_size = 0.2, random_state = 42)

In [59]:
# Apply Simple Impuer and OneHotEncoder using pipeline
new_num_trf = Pipeline([('imputer', SimpleImputer(strategy = 'median'))])
new_cat_trf = Pipeline([('imputer', SimpleImputer(strategy = 'most_frequent')),
                       ('encoder', OneHotEncoder(handle_unknown = 'ignore'))])

# Use Column Transformer
new_preprocessor = ColumnTransformer([
    ('num', new_num_trf, new_num_col),
    ('cat', new_cat_trf, new_cat_col)
])

In [60]:
# Apply Model And Transformer using Pipeline
new_model4 = Pipeline([
    ('preprocessor', new_preprocessor),
    ('model', GradientBoostingRegressor(n_estimators=100,learning_rate=0.1,max_depth=4,random_state=42))
])

In [61]:
# train Model
new_model4.fit(new_x_train, new_y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [62]:
# Predict Model
new_y_pred4 = new_model4.predict(new_x_test)

In [63]:
# Check R2 Score
r2_score(new_y_test, new_y_pred4)

0.7843348117366077

In [64]:
# Cross validation using r2 score
score = cross_val_score(new_model4, new_x, new_y, cv = 10, scoring = 'r2')
print('Cross validation of r2 score:', score.mean())

Cross validation of r2 score: 0.7172100052393363


In [65]:
# Cross Validation Using Root Mean Square Error
score = cross_val_score(new_model4, new_x, new_y, cv = 10, scoring = 'neg_root_mean_squared_error')
print('root_mean_square_error:', -score.mean())

root_mean_square_error: 41673.05994416075


In [66]:
# Apply Model and Transformer using pipeline
new_model3 = Pipeline([
    ('preprocessor', new_preprocessor),
    ('model', LinearRegression())
])

In [67]:
# Model Train
new_model3.fit(new_x_train, new_y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [68]:
# Predict Model
new_y_pred3 = new_model3.predict(new_x_test)

In [69]:
# Check r2 score
r2_score(new_y_test, new_y_pred3)

0.7558702909567888

In [70]:
# cross Validation using r2 score
score = cross_val_score(new_model3, new_x, new_y, cv = 10, scoring = 'r2')
print('Cross validation of r2 score:', score.mean())

Cross validation of r2 score: 0.6883101483223286


In [71]:
# Cross Validation Using Root Mean Square Error
score = cross_val_score(new_model3, new_x, new_y, cv = 10, scoring = 'neg_root_mean_squared_error')
print('root_mean_square_error:', -score.mean())

root_mean_square_error: 43944.99054424579


In [72]:
# Aplly Model and Transformer using Pipeline
new_model2 = Pipeline([
    ('preprocessor', new_preprocessor),
    ('model', XGBRegressor(n_estimators=100,learning_rate=0.1,max_depth=4,random_state=42))
])

In [73]:
# Train Model
new_model2.fit(new_x_train, new_y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [74]:
# Predict Model
new_y_pred2 = new_model2.predict(new_x_test)

In [75]:
# check R2 score 
r2_score(new_y_test, new_y_pred2)

0.7763168215751648

In [76]:
# Cross validation Using R2 score
score = cross_val_score(new_model2, new_x, new_y, cv = 10, scoring = 'r2')
print('Cross validation of r2 score:', score.mean())

Cross validation of r2 score: 0.7267623782157898


In [77]:
# Cross Validation Using Root Mean Square Error
score = cross_val_score(new_model2, new_x, new_y, cv = 10, scoring = 'neg_root_mean_squared_error')
print('root_mean_square_error:', -score.mean())

root_mean_square_error: 40938.410546875


In [78]:
# Aplly Model and Transformer using pipeline
new_model1 = Pipeline([
    ('preprocessor', new_preprocessor),
    ('model',RandomForestRegressor(n_estimators=100))
])

In [79]:
# train Model
new_model1.fit(new_x_train, new_y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [80]:
# Predict Model
new_y_pred1 = new_model1.predict(new_x_test)

In [81]:
# Check R2 score
r2_score(new_y_test, new_y_pred1)

0.7601765014076468

In [82]:
# Cross Validation using R2 score
score = cross_val_score(new_model1, new_x, new_y, cv = 10, scoring = 'r2')
print('Cross validation of r2 score:', score.mean())

Cross validation of r2 score: 0.7095051315227806


In [83]:
# Cross Validation Using Root Mean Square Error
score = cross_val_score(new_model1, new_x, new_y, cv = 10, scoring = 'neg_root_mean_squared_error')
print('root_mean_square_error:', -score.mean())

root_mean_square_error: 41953.71261322177


In [84]:
df.head(2)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500


In [85]:
df.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

# I will take 7 most important and user friendly feature to predict the saleprice

# LotArea
# HouseStyle
# Neighborhood
# YearBuilt
# OverallQual
# TotalBsmtSF
# GarageCars

In [86]:
# Define Input and Output
xx = df[['LotArea','HouseStyle','Neighborhood','YearBuilt','OverallQual','TotalBsmtSF','GarageCars']]
yy = df['SalePrice']

In [87]:
xx

,LotArea,HouseStyle,Neighborhood,YearBuilt,OverallQual,TotalBsmtSF,GarageCars
0,8450,2Story,CollgCr,2003,7,856,2
1,9600,1Story,Veenker,1976,6,1262,2
2,11250,2Story,CollgCr,2001,7,920,2
3,9550,2Story,Crawfor,1915,7,756,3
4,14260,2Story,NoRidge,2000,8,1145,3
...,...,...,...,...,...,...,...
1455,7917,2Story,Gilbert,1999,6,953,2
1456,13175,1Story,NWAmes,1978,6,1542,2
1457,9042,2Story,Crawfor,1941,7,1152,1
1458,9717,1Story,NAmes,1950,5,1078,1


In [88]:
yy

0       208500
1       181500
2       223500
3       140000
4       250000
         ...  
1455    175000
1456    210000
1457    266500
1458    142125
1459    147500
Name: SalePrice, Length: 1460, dtype: int64

In [89]:
# Check null
xx.isnull().sum()

LotArea         0
HouseStyle      0
Neighborhood    0
YearBuilt       0
OverallQual     0
TotalBsmtSF     0
GarageCars      0
dtype: int64

In [90]:
# check null
yy.isnull().sum()

np.int64(0)

In [91]:
# Train Test Split
xx_train, xx_test, yy_train, yy_test = train_test_split(xx, yy, test_size=0.2, random_state=42)

In [92]:
# Preprocessing
transform = ColumnTransformer([
    ('encoder',OneHotEncoder(drop='first', handle_unknown='ignore'),['HouseStyle', 'Neighborhood'])
], remainder='passthrough')

In [93]:
# Models
models = {
    'XGBoost': XGBRegressor(random_state=42)
}

In [94]:
for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', transform),
        ('model', model)
    ])

### XGBoostRegressor

In [95]:
pipeline.fit(xx_train, yy_train)
pred = pipeline.predict(xx_test)
r2 = r2_score(yy_test, pred)
print(f"{name}: {r2:.4f}")

XGBoost: 0.8587


In [96]:
score = cross_val_score(pipeline,xx,yy,cv=10,scoring='r2')
print(f"{name} CV R²: {score.mean():.4f}")

XGBoost CV R²: 0.8271


In [97]:
import joblib

joblib.dump(pipeline, "xgb_model.pkl")

['xgb_model.pkl']

In [98]:
# Models
models1 = {
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

In [99]:
for name, model in models1.items():
    pipeline1 = Pipeline([
        ('preprocessor', transform),
        ('model', model)
    ])

### Gradient Boosting Regressor

In [100]:
pipeline1.fit(xx_train, yy_train)
pred1 = pipeline1.predict(xx_test)
r2 = r2_score(yy_test, pred1)
print(f"{name}: {r2:.4f}")

Gradient Boosting: 0.8825


In [101]:
score = cross_val_score(pipeline1,xx,yy,cv=10,scoring='r2')
print(f"{name} CV R²: {score.mean():.4f}")

Gradient Boosting CV R²: 0.8447


In [102]:
import joblib

joblib.dump(pipeline, "gb_model.pkl")

['gb_model.pkl']

In [103]:
# Models
models2 = {
    'Random Forest': RandomForestRegressor(random_state=42)
}

In [104]:
for name, model in models2.items():
    pipeline2 = Pipeline([
        ('preprocessor', transform),
        ('model', model)
    ])

### Random Forest Regressor

In [105]:
pipeline2.fit(xx_train, yy_train)
pred2 = pipeline2.predict(xx_test)
r2 = r2_score(yy_test, pred2)
print(f"{name}: {r2:.4f}")

Random Forest: 0.8650


In [106]:
score = cross_val_score(pipeline2,xx,yy,cv=10,scoring='r2')
print(f"{name} CV R²: {score.mean():.4f}")

Random Forest CV R²: 0.8179


In [107]:
# Models
models3 = {
    'Linear Regression': LinearRegression()
}

In [108]:
for name, model in models3.items():
    pipeline3 = Pipeline([
        ('preprocessor', transform),
        ('model', model)
    ])

In [ ]:
### linear Regress

In [109]:
pipeline3.fit(xx_train, yy_train)
pred3 = pipeline3.predict(xx_test)
r2 = r2_score(yy_test, pred3)
print(f"{name}: {r2:.4f}")

Linear Regression: 0.7951


In [110]:
score = cross_val_score(pipeline3,xx,yy,cv=10,scoring='r2')
print(f"{name} CV R²: {score.mean():.4f}")

Linear Regression CV R²: 0.7624
